# 24 — Post processing

Helpers that decorate atlas outputs with country context so downstream analyses can group / filter by country or continent. Also adds raw data of _apparent_temp_monthly.nc, _precipitation_monthly.nc and _population_density.nc

In [6]:
import geopandas as gpd
import httpx
import pandas as pd
import xarray as xr

from common import PROCESSED_DIR, RAW_DIR

variable_raw = RAW_DIR / 'post_processing'
variable_raw.mkdir(parents=True, exist_ok=True)

## 1. Country polygons

Natural Earth 50m admin_0 boundaries — same shapefile used by `18_natural_disaster_risk` and `19_climate_vulnerability`. Loaded here directly (not via `regionmask`) for the `ISO_A3_EH` and `CONTINENT` attributes.

In [7]:
NE_URL = 'https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip'
ne_zip = variable_raw / 'ne_50m_admin_0_countries.zip'

if not ne_zip.exists():
    print(f'downloading {NE_URL}')
    r = httpx.get(NE_URL, headers={'User-Agent': 'Mozilla/5.0'}, timeout=180, follow_redirects=True)
    r.raise_for_status()
    ne_zip.write_bytes(r.content)

print(f'{ne_zip.name}: {ne_zip.stat().st_size / 1024**2:.1f} MB')

ne_50m_admin_0_countries.zip: 0.8 MB


In [8]:
countries = (
    gpd.read_file(f'zip://{ne_zip}')
    .to_crs('EPSG:4326')
    [['ISO_A3_EH', 'CONTINENT', 'geometry']]
    .rename(columns={'ISO_A3_EH': 'country_code', 'CONTINENT': 'continent'})
)
countries.head()

,country_code,continent,geometry
0,ZWE,Africa,"POLYGON ((31.28789 -22.40205, 31.19727 -22.344..."
1,ZMB,Africa,"POLYGON ((30.39609 -15.64307, 30.25068 -15.643..."
2,YEM,Asia,"MULTIPOLYGON (((53.08564 16.64839, 52.58145 16..."
3,VNM,Asia,"MULTIPOLYGON (((104.06396 10.39082, 104.08301 ..."
4,VEN,South America,"MULTIPOLYGON (((-60.82119 9.13838, -60.94141 9..."


## 2. Enrich a dataframe

Point-in-polygon join against the Natural Earth polygons. Rows whose point falls in the ocean or in a disputed area not covered by any polygon get NaN for both columns.

In [9]:
def add_country_and_continent(df, lat_col='lat', lon_col='lon', polygons=None):
    """Enrich ``df`` with ``country_code`` (ISO3) and ``continent`` columns.

    Each row's ``(lat, lon)`` is looked up against Natural Earth 50m admin_0
    country polygons via a spatial join. Rows whose point falls outside every
    polygon (ocean cells, unclaimed territory) get NaN in both columns.

    Parameters
    ----------
    df : pandas.DataFrame
        Input frame with latitude and longitude columns in EPSG:4326.
    lat_col, lon_col : str
        Column names holding the coordinates.
    polygons : geopandas.GeoDataFrame, optional
        Country polygons with ``country_code`` and ``continent`` columns.
        Defaults to the ``countries`` frame loaded above.

    Returns
    -------
    pandas.DataFrame
        Copy of ``df`` with two extra columns appended. Row order preserved.
    """
    polys = countries if polygons is None else polygons
    points = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[lon_col], df[lat_col]),
        crs='EPSG:4326',
    )
    joined = points.sjoin(
        polys[['country_code', 'continent', 'geometry']],
        how='left',
        predicate='within',
    )
    # sjoin can duplicate rows when a point sits exactly on a shared border;
    # keep the first hit so output length matches input length.
    joined = joined[~joined.index.duplicated(keep='first')]
    return pd.DataFrame(joined.drop(columns=['geometry', 'index_right']))

## 3. Enrich the normalized atlas

Load `normalized.nc` (all percentile-transformed layers from `92_normalization`), flatten the `(lat, lon)` grid into rows, attach `country_code` and `continent`, and save as `normalized_by_country.parquet` for downstream country / continent aggregations.

In [ ]:
ds = xr.open_dataset(PROCESSED_DIR / 'normalized.nc')
df = ds.to_dataframe().reset_index()

# Cells that are NaN in every layer are pure ocean or coverage gaps — they
# have nothing to aggregate later, so drop before the (expensive) spatial join.
layer_cols = [c for c in df.columns if c not in ('lat', 'lon')]
df = df.dropna(subset=layer_cols, how='all').reset_index(drop=True)

df = add_country_and_continent(df)

n_missing = df['country_code'].isna().sum()
print(f'{len(df):,} cells; {n_missing:,} ({n_missing / len(df):.1%}) with no country match')

df = df.rename(columns={"country_code": "_country_code", "continent" : "_continent"})

out = PROCESSED_DIR / 'normalized_wc.parquet'
df.to_parquet(out, index=False)
print(f'wrote {out}')
df.head()

85,959 cells; 1,479 (1.7%) with no country match
wrote /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/processed/normalized_by_country.parquet


,lat,lon,sea_proximity,terrain_ruggedness,sun_hours,temperature_pleasantness,annual_greenness,precipitation_balance,climate_vulnerability,natural_disaster_risk,...,urbanity,internet_connectivity,healthcare_access,income,cost_of_living,crime_rate,human_freedom,corruption,_country_code,_continent
0,-89.75,-179.75,0.821762,NaN,NaN,0.200856,NaN,0.148184,NaN,NaN,...,0.19171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATA,Antarctica
1,-89.75,-179.25,0.820982,NaN,NaN,0.200856,NaN,0.148184,NaN,NaN,...,0.19171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATA,Antarctica
2,-89.75,-178.75,0.823332,NaN,NaN,0.200856,NaN,0.148184,NaN,NaN,...,0.19171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATA,Antarctica
3,-89.75,-178.25,0.826869,NaN,NaN,0.200856,NaN,0.148184,NaN,NaN,...,0.19171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATA,Antarctica
4,-89.75,-177.75,0.825868,NaN,NaN,0.200856,NaN,0.148184,NaN,NaN,...,0.19171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATA,Antarctica


## 4. Raw scoring inputs

Compact land-only parquet holding the three raw layers that `99_atlas_map.ipynb` re-scores on the fly under different climate / rainfall / density preferences: monthly apparent temperature, monthly precipitation, and population density. Ocean cells and rows outside every country polygon are dropped so the file is small enough to load quickly instead of pulling ~25 MB of gridded NetCDFs.

Layout is wide — one row per `(lat, lon)`, with 12 columns per monthly variable (`apparent_temp_01`..`apparent_temp_12`, `precipitation_01`..`precipitation_12`) and a scalar `population_density`. `common.load_raw_scoring_inputs` reconstructs the three DataArrays on the shared atlas grid.

In [11]:
def _monthly_to_wide(da, prefix):
    """Unstack a ``(lat, lon, month)`` DataArray into a ``{prefix}_MM`` wide frame."""
    return (
        da.to_dataframe(name=prefix)
        .unstack('month')[prefix]
        .rename(columns=lambda m: f'{prefix}_{int(m):02d}')
    )

at_da = xr.open_dataarray(PROCESSED_DIR / '_apparent_temp_monthly.nc')
precip_da = xr.open_dataarray(PROCESSED_DIR / '_precipitation_monthly.nc')
density_da = xr.open_dataarray(PROCESSED_DIR / '_population_density.nc')

raw = (
    _monthly_to_wide(at_da, 'apparent_temp')
    .join(_monthly_to_wide(precip_da, 'precipitation'), how='outer')
    .join(density_da.to_dataframe(name='population_density'), how='outer')
    .reset_index()
)

raw = add_country_and_continent(raw)
raw = raw[raw['country_code'].notna()].drop(columns=['country_code', 'continent'])

value_cols = [c for c in raw.columns if c not in ('lat', 'lon')]
raw = raw.dropna(subset=value_cols, how='all').reset_index(drop=True)

out = PROCESSED_DIR / 'raw_scoring_inputs.parquet'
raw.to_parquet(out, index=False)
print(f'wrote {out} ({out.stat().st_size / 1024**2:.1f} MB, {len(raw):,} rows)')
raw.head()

wrote /Users/lutz/Documents/alfatraining/projects/data-science-notebooks/data/world-livable-atlas/processed/raw_scoring_inputs.parquet (7.9 MB, 84,480 rows)


,lat,lon,apparent_temp_01,apparent_temp_02,apparent_temp_03,apparent_temp_04,apparent_temp_05,apparent_temp_06,apparent_temp_07,apparent_temp_08,...,precipitation_04,precipitation_05,precipitation_06,precipitation_07,precipitation_08,precipitation_09,precipitation_10,precipitation_11,precipitation_12,population_density
0,-89.75,-179.75,-32.853405,-44.560394,-56.275181,-60.583542,-61.367447,-62.874058,-65.388313,-64.790833,...,7.2,6.51,6.3,5.115,5.115,6.75,5.89,4.05,4.185,0.0
1,-89.75,-179.25,-32.850357,-44.560394,-56.276184,-60.586544,-61.368450,-62.875061,-65.392319,-64.791840,...,7.2,6.51,6.3,5.115,5.115,6.75,5.89,4.05,4.185,0.0
2,-89.75,-178.75,-32.850357,-44.560394,-56.280193,-60.586544,-61.372456,-62.879063,-65.396324,-64.795837,...,7.2,6.51,6.3,5.115,5.115,6.75,5.89,4.05,4.185,0.0
3,-89.75,-178.25,-32.850357,-44.560394,-56.280193,-60.590549,-61.372456,-62.883068,-65.396324,-64.799843,...,7.2,6.51,6.3,5.115,5.115,6.75,5.89,4.05,4.185,0.0
4,-89.75,-177.75,-32.850365,-44.560398,-56.280197,-60.591549,-61.375458,-62.884071,-65.399323,-64.803848,...,7.2,6.51,6.3,5.115,5.115,6.75,5.89,4.05,4.185,0.0
